# Semantic Categories #

This Notebook will be used to check our available labels, and decide on a set of semantic categories that will be used for training. 

In [3]:
import numpy as np
import nltk
from nltk.corpus import wordnet as wn
from collections import defaultdict

# Download WordNet data (only needed once)
try:
    wn.synsets('test')
except LookupError:
    print("Downloading WordNet data...")
    nltk.download('wordnet')
    nltk.download('omw-1.4')
    print("✓ Download complete")

[nltk_data] Downloading package wordnet to /Users/mario/nltk_data...
[nltk_data] Downloading package omw-1.4 to /Users/mario/nltk_data...


✓ Download complete


### WordNet Basics - Exploring Synsets and Hypernyms

Let's explore how WordNet represents words and their hierarchies

In [4]:
# Example 1: Single word - "dog"
word = "dog"
synsets = wn.synsets(word, pos=wn.NOUN)  # Get all noun synsets for "dog"

print(f"Word: '{word}'")
print(f"Number of synsets (meanings): {len(synsets)}\n")

# Let's use the first (most common) synset
synset = synsets[0]
print(f"Synset: {synset}")
print(f"Definition: {synset.definition()}")
print(f"Examples: {synset.examples()}")
print(f"\nHypernym path (from specific to general):")

# Get hypernym path - shows the hierarchy
paths = synset.hypernym_paths()
for i, path in enumerate(paths[0]):  # Show first path
    print(f"  {i}. {path.name().split('.')[0]:20s} - {path.definition()[:60]}")

Word: 'dog'
Number of synsets (meanings): 7

Synset: Synset('dog.n.01')
Definition: a member of the genus Canis (probably descended from the common wolf) that has been domesticated by man since prehistoric times; occurs in many breeds
Examples: ['the dog barked all night']

Hypernym path (from specific to general):
  0. entity               - that which is perceived or known or inferred to have its own
  1. physical_entity      - an entity that has physical existence
  2. object               - a tangible and visible entity; an entity that can cast a sha
  3. whole                - an assemblage of parts that is regarded as a single entity
  4. living_thing         - a living (or once living) entity
  5. organism             - a living thing that has (or can develop) the ability to act 
  6. animal               - a living organism characterized by voluntary movement
  7. domestic_animal      - any of various animals that have been tamed and made fit for
  8. dog                  - a m

In [ ]:
# Example 2: Multi-word term from our data - "Tibetan_mastiff"
word = "Tibetan_mastiff"

# Try as-is
synsets = wn.synsets(word, pos=wn.NOUN)
print(f"Searching for: '{word}'")
print(f"  Found {len(synsets)} synsets directly\n")


if synsets:
    synset = synsets[0]
    print(f"Synset: {synset}")
    print(f"Definition: {synset.definition()}\n")
    
    # Get just the hypernyms (parent concepts)
    print("Direct hypernyms (one level up):")
    for hyp in synset.hypernyms():
        print(f"  → {hyp.name().split('.')[0]:20s} - {hyp.definition()[:60]}")


Searching for: 'Tibetan_mastiff'
  Found 1 synsets directly

Synset: Synset('tibetan_mastiff.n.01')
Definition: very large powerful rough-coated dog native to central Asia

Direct hypernyms (one level up):
  → mastiff              - an old breed of powerful deep-chested smooth-coated dog used


## LESSON ##
Use underscores not spaces 

### Main Function: Extract Hierarchies from Labels

In [20]:
def get_hierarchical_categories(word, max_depth=6, verbose=False):
    """
    Extract hierarchical categories for a given word using WordNet.
    
    Returns:
    --------
    dict with keys:
        'word': original word
        'found': bool - whether word was found in WordNet
        'synset': the synset used (first/most common)
        'definition': synset definition
        'hierarchy': list of category names from specific to general

    """
    result = {
        'word': word,
        'found': False,
        'synset': None,
        'definition': None,
        'hierarchy': [],

    }
    
    # Preprocessing: handle underscores, lowercase
    word_variations = [
        word,
        word.replace('_', ' '),
        word.replace('_', ' ').lower(),
        word.lower(),
    ]
    
    # Try to find synsets
    synsets = []
    for variant in word_variations:
        synsets = wn.synsets(variant, pos=wn.NOUN)
        if synsets:
            if verbose:
                print(f"✓ Found '{variant}' in WordNet")
            break
    
    if not synsets:
        if verbose:
            print(f"✗ '{word}' not found in WordNet")
        return result
    
    # Use the first (most common) synset
    synset = synsets[0]
    result['found'] = True
    result['synset'] = synset.name()
    result['definition'] = synset.definition()
    
    # Get hypernym path (from specific to general)
    paths = synset.hypernym_paths()
    if paths:
        # Take the first path
        path = paths[0]
        
        # Extract clean category names (remove .n.01 suffixes)
        hierarchy = [s.name().split('.')[0].replace('_', ' ') for s in path]
        result['hierarchy'] = hierarchy
    
    
    return result

### Test the Function with Real Labels

Let's test our function with actual labels from the dataset

In [28]:
# Test with examples from our dataset
test_words = [
    "Tibetan_mastiff",  # ImageNet - dog breed
    "banana",           # COCO - food
    "pretzel",          # ImageNet - food
    "brain_coral",      # ImageNet - marine organism
    "laptop",           # COCO - electronics
    "nematode",         # ImageNet - worm
    "carousel",         # ImageNet - structure
    "hammer",
    "pencil",
    "bag",
    "camera",
    "cigar",
    "chair",
    "beach",
    "road",
    "phone",
    "red",
    "Banquet Hall"
]

print("=" * 80)
for word in test_words:
    result = get_hierarchical_categories(word, verbose=False)
    
    print(f"\nWord: {word}")
    if result['found']:
        print(f"  Hierarchy: {' → '.join(result['hierarchy'][::-1])}")  # Reverse to show specific→general
    else:
        print(f"   Not found in WordNet")
    print("-" * 80)


Word: Tibetan_mastiff
  Hierarchy: tibetan mastiff → mastiff → working dog → dog → domestic animal → animal → organism → living thing → whole → object → physical entity → entity
--------------------------------------------------------------------------------

Word: banana
  Hierarchy: banana → herb → vascular plant → plant → organism → living thing → whole → object → physical entity → entity
--------------------------------------------------------------------------------

Word: pretzel
  Hierarchy: pretzel → cracker → bread → baked goods → food → solid → matter → physical entity → entity
--------------------------------------------------------------------------------

Word: brain_coral
  Hierarchy: brain coral → stony coral → coral → anthozoan → coelenterate → invertebrate → animal → organism → living thing → whole → object → physical entity → entity
--------------------------------------------------------------------------------

Word: laptop
  Hierarchy: laptop → portable computer →

## Hand Picked Categories ##

In [ ]:
categories_wordnet_firstpass = ["person", "animal", "food", "vehicle", "location", "tool", "furniture", "color",
                            
"feeling", "establishment", "structure", "clothing", "living thing"]

aliasesNsubcategories = {
    "location": ["scene", "geological formation", "room", "station", "path", "road" , "place"],
    "tool": ["instrumentality"],
    "person": ["people"],
    "establishment": ["shop"],
    "food": ["beverage", "edible", "eat"],
    "feeling": ["sensation"],
    "clothing": ["garment", "footwear", "hat", "headdress"],
    "animal": [],
    "vehicle": ["transportation"],
    "plant": [],
    "furniture": [],
    "color": [],
    "structure": ["room"],
    "living thing": ["organic", "body part"]
}

### Map Labels to Categories

Process all single-label entries from our dataset and map them to the categories above.

In [59]:
# Load all labels from processed data (single-label only)
data = np.load('processed_data/CSI1/CSI1_schaefer400.npz', allow_pickle=True)
labels = data['labels']

# Extract only single-label entries
single_labels = []
for lbl_list in labels:
    if len(lbl_list) == 1:
        single_labels.append(lbl_list[0])

# Get unique labels
unique_labels = sorted(set(single_labels))
print(f"Total single-label entries: {len(single_labels)}")
print(f"Unique single labels: {len(unique_labels)}")

# Categories to map to (normalize for matching)
categories = categories_wordnet_firstpass


# Map each label to a category
label_to_category = {}  # {label: category}
label_to_hierarchy = {}  # {label: full hierarchy list}
matched_labels = []
unmatched_labels = []

print("\nProcessing labels...")
for i, label in enumerate(unique_labels):
    if (i + 1) % 50 == 0:
        print(f"  Processed {i + 1}/{len(unique_labels)}...")
    
    result = get_hierarchical_categories(label, verbose=False)
    
    if not result['found']:
        unmatched_labels.append(label)
        continue
    
    # Store the hierarchy
    label_to_hierarchy[label] = result['hierarchy']
    
    # Try to match to one of our categories
    matched = False
    for concept in result['hierarchy']:
        concept_lower = concept.lower()
        
        # Direct match to category
        if concept_lower in categories:
            label_to_category[label] = concept_lower
            matched_labels.append(label)
            matched = True
            break
        
        # Check individual words in multi-word concepts
        concept_words = concept_lower.split()
        if len(concept_words) > 1:
            for word in concept_words:
                if word in categories:
                    label_to_category[label] = word
                    matched_labels.append(label)
                    matched = True
                    break
        
        if matched:
            break
        
        # Check if concept matches any category's aliases
        for category, aliases in aliasesNsubcategories.items():
            if concept_lower in aliases:
                label_to_category[label] = category
                matched_labels.append(label)
                matched = True
                break
        
        if matched:
            break
    
    if not matched:
        unmatched_labels.append(label)

print(f"\n✓ Processing complete!")
print(f"  Matched:   {len(matched_labels)}/{len(unique_labels)} ({100*len(matched_labels)/len(unique_labels):.1f}%)")
print(f"  Unmatched: {len(unmatched_labels)}/{len(unique_labels)} ({100*len(unmatched_labels)/len(unique_labels):.1f}%)")

Total single-label entries: 3792
Unique single labels: 1244

Processing labels...
  Processed 50/1244...
  Processed 100/1244...
  Processed 150/1244...
  Processed 200/1244...
  Processed 250/1244...
  Processed 300/1244...
  Processed 350/1244...
  Processed 400/1244...
  Processed 450/1244...
  Processed 500/1244...
  Processed 550/1244...
  Processed 600/1244...
  Processed 650/1244...
  Processed 700/1244...
  Processed 750/1244...
  Processed 800/1244...
  Processed 850/1244...
  Processed 900/1244...
  Processed 950/1244...
  Processed 1000/1244...
  Processed 1050/1244...
  Processed 1100/1244...
  Processed 1150/1244...
  Processed 1200/1244...

✓ Processing complete!
  Matched:   955/1244 (76.8%)
  Unmatched: 289/1244 (23.2%)


### Results: Matched Labels by Category

In [60]:
# Group matched labels by category
from collections import defaultdict
category_to_labels = defaultdict(list)

for label, category in label_to_category.items():
    category_to_labels[category].append(label)

# Display breakdown
print("=" * 80)
print("MATCHED LABELS BY CATEGORY")
print("=" * 80)

for category in sorted(category_to_labels.keys()):
    labels_in_cat = category_to_labels[category]
    print(f"\n{category.upper()} ({len(labels_in_cat)} labels):")
    # Show first 10 examples
    for i, label in enumerate(sorted(labels_in_cat)[:10], 1):
        print(f"  {i}. {label}")


print("\n" + "=" * 80)

MATCHED LABELS BY CATEGORY

ANIMAL (5 labels):
  1. bearskin
  2. beaver
  3. leopard
  4. mink
  5. otter

CLOTHING (44 labels):
  1. Cardigan
  2. Christmas_stocking
  3. Windsor_tie
  4. abaya
  5. academic_gown
  6. apron
  7. bathing_cap
  8. bolo_tie
  9. bonnet
  10. bow_tie

FEELING (2 labels):
  1. Amusement
  2. perfume

FOOD (34 labels):
  1. American_lobster
  2. Dungeness_crab
  3. French_loaf
  4. Icecream
  5. bagel
  6. burrito
  7. carbonara
  8. cheeseburger
  9. chocolate_sauce
  10. coho

LIVING THING (366 labels):
  1. Afghan_hound
  2. African_chameleon
  3. African_crocodile
  4. African_elephant
  5. African_grey
  6. African_hunting_dog
  7. Airedale
  8. American_Staffordshire_terrier
  9. American_alligator
  10. American_black_bear

LOCATION (37 labels):
  1. Airport_Terminal
  2. Angora
  3. Backyard
  4. Bakery
  5. Beach
  6. Bleachers
  7. Boardwalk
  8. Campsite
  9. Canyon
  10. Cave

PERSON (43 labels):
  1. Dentist
  2. Diner
  3. Doctor
  4. Loafer


### Unmatched Labels

These labels were found in WordNet but didn't match any of our categories. Review their hierarchies to see what categories we should add.

In [61]:
# Show unmatched labels with their hierarchies
print("=" * 80)
print(f"UNMATCHED LABELS ({len(unmatched_labels)} total)")
print("=" * 80)

# Separate into: found in WordNet but not matched, vs not found in WordNet
unmapped_with_hierarchy = []
not_in_wordnet = []

for label in unmatched_labels:
    if label in label_to_hierarchy:
        unmapped_with_hierarchy.append(label)
    else:
        not_in_wordnet.append(label)

print(f"\nFound in WordNet but didn't match categories: {len(unmapped_with_hierarchy)}")
print(f"Not found in WordNet at all: {len(not_in_wordnet)}\n")

# Show unmapped labels with their hierarchies (limit to 20 for readability)
if unmapped_with_hierarchy:
    print("-" * 80)
    print("LABELS WITH HIERARCHIES (but no category match):")
    print("Review these to identify missing categories")
    print("-" * 80)
    
    for i, label in enumerate(sorted(unmapped_with_hierarchy), 1):
        hierarchy = label_to_hierarchy[label]
        # Show last 4 levels of hierarchy (most relevant)
        relevant_hierarchy = hierarchy
        print(f"{i:2d}. {label:25s} → {' → '.join(relevant_hierarchy[::-1])}")
    

# Show labels not found in WordNet
if not_in_wordnet:
    print("\n" + "-" * 80)
    print("NOT IN WORDNET (need manual handling):")
    print("-" * 80)
    for i, label in enumerate(sorted(not_in_wordnet), 1):
        print(f"{i:2d}. {label}")

print("\n" + "=" * 80)

UNMATCHED LABELS (289 total)

Found in WordNet but didn't match categories: 156
Not found in WordNet at all: 133

--------------------------------------------------------------------------------
LABELS WITH HIERARCHIES (but no category match):
Review these to identify missing categories
--------------------------------------------------------------------------------
 1. Alley                     → alley → street → thoroughfare → road → way → artifact → whole → object → physical entity → entity
 2. Arcade                    → arcade → passageway → passage → way → artifact → whole → object → physical entity → entity
 3. Atm                       → standard atmosphere → pressure unit → unit of measurement → definite quantity → measure → abstraction → entity
 4. Backstage                 → wing → stage → platform → horizontal surface → surface → artifact → whole → object → physical entity → entity
 5. Band_Aid                  → band aid → adhesive bandage → bandage → dressing → cloth cove

### Save Results for Later Review

In [62]:
import json

semantic_mapping = {}

for category in categories_wordnet_firstpass:
    semantic_mapping[category] = {
        "aliases": aliasesNsubcategories.get(category, []),
        "labels": sorted(category_to_labels.get(category, []))
    }

with open('SemanticLabels.json', 'w') as f:
    json.dump(semantic_mapping, f, indent=2)

print("✓ Saved semantic mapping to SemanticLabels.json")
print(f"\nStructure:")
for category, data in semantic_mapping.items():
    print(f"  {category}: {len(data['labels'])} labels, {len(data['aliases'])} aliases")

✓ Saved semantic mapping to SemanticLabels.json

Structure:
  person: 43 labels, 1 aliases
  animal: 5 labels, 0 aliases
  food: 34 labels, 3 aliases
  vehicle: 0 labels, 1 aliases
  location: 37 labels, 5 aliases
  tool: 316 labels, 1 aliases
  furniture: 0 labels, 0 aliases
  color: 0 labels, 0 aliases
  feeling: 2 labels, 1 aliases
  establishment: 0 labels, 1 aliases
  structure: 108 labels, 1 aliases
  clothing: 44 labels, 4 aliases
  living thing: 366 labels, 2 aliases


#LLM powered labeling. 
We've done the programatic thing, which has many flaws.
Now we will ask LLM to manually sort into best categories but we will give it a structure to do so that it reduces hallucinations

We will ask it to create a set of raw labels before, then after, and then this set should be of the same before and after, we should have the eact same labels, just in different places. 

In [64]:
def get_all_labels_from_semantic_json(json_path='SemanticLabels.json'):
    """
    Extract all labels from SemanticLabels.json into a set.
    """
    with open(json_path, 'r') as f:
        semantic_data = json.load(f)
    
    all_labels = set()
    for category, data in semantic_data.items():
        labels_list = data.get('labels', [])
        all_labels.update(labels_list)
    
    return all_labels

labels_before = get_all_labels_from_semantic_json('SemanticLabels.json')
print(f"✓ Extracted {len(labels_before)} unique labels from SemanticLabels.json")
print(f"Sample labels: {list(labels_before)[:10]}")

✓ Extracted 955 unique labels from SemanticLabels.json
Sample labels: ['moving_van', 'pencil_box', 'bicycle-built-for-two', 'indigo_bunting', 'sorrel', 'panpipe', 'croquet_ball', 'slug', 'Polaroid_camera', 'aircraft_carrier']


In [65]:
# Step 2: Manual corrections - identifying misclassified labels

# Load current semantic labels
with open('SemanticLabels.json', 'r') as f:
    semantic_data = json.load(f)

# Create correction mappings: label -> correct_category
corrections = {}

# Animals incorrectly in "person"
animals_from_person = ["Samoyed", "badger", "crane", "dalmatian", "guinea_pig", "hammerhead", 
                       "hog", "jay", "loggerhead", "monarch", "monitor", "ostrich", "pirate",
                       "skunk", "tiger", "weasel"]

# Objects/tools incorrectly in "person"
tools_from_person = ["bannister", "doormat", "drake", "harvester", "hot_dog", "hotdog",
                     "pitcher", "plunger", "printer", "punching_bag", "sax", "screw", 
                     "toaster", "washer"]

# Food items incorrectly in "person"
food_from_person = ["hot_dog", "hotdog"]

# Clothing from person
clothing_from_person = ["Loafer", "groom"]

# Dog breeds incorrectly in "location"
animals_from_location = ["Angora", "Chihuahua", "Lhasa"]

# Vehicles currently in "tool" that should be "vehicle"
vehicles_from_tool = ["aircraft_carrier", "airliner", "airplane", "airship", "ambulance", "amphibian",
                      "beach_wagon", "bicycle-built-for-two", "boat", "bobsled", "bullet_train", "bus",
                      "canoe", "car", "car_mirror", "car_wheel", "catamaran", "container_ship", 
                      "convertible", "dogsled", "fire_engine", "fireboat", "forklift", "four-poster",
                      "freight_car", "garbage_truck", "go-kart", "golfcart", "gondola", "half_track",
                      "horse_cart", "jeep", "jinrikisha", "lifeboat", "limousine", "minibus", "minivan",
                      "missile", "moped", "motor_scooter", "motorcycle", "mountain_bike", "moving_van",
                      "oxcart", "passenger_car", "pickup", "plane", "police_van", "projectile",
                      "recreational_vehicle", "school_bus", "schooner", "snowmobile", "snowplow",
                      "space_shuttle", "speedboat", "sports_car", "steam_locomotive", "streetcar",
                      "submarine", "tank", "tow_truck", "tractor", "trailer_truck", "train",
                      "tricycle", "trimaran", "trolleybus", "truck", "unicycle", "warplane", "yawl"]

# Furniture from "tool"
furniture_from_tool = ["barber_chair", "bed", "bench", "bookcase", "china_cabinet", "chiffonier",
                       "cradle", "crib", "desk", "dining_table", "entertainment_center", "folding_chair",
                       "four-poster", "park_bench", "rocking_chair", "studio_couch", "throne", "wardrobe"]

# Structures/buildings from "tool"
structures_from_tool = ["altar", "cinema", "drilling_platform", "flagpole", "home_theater", "toilet"]

# Food/plants from "structure"
food_from_structure = ["Granny_Smith", "acorn", "buckeye", "lemon", "orange", "rapeseed", "strawberry"]

# Locations from structure
location_from_structure = ["dam", "patio"]

# Apply corrections
for label in animals_from_person + animals_from_location:
    corrections[label] = "living thing"

for label in tools_from_person:
    if label not in food_from_person and label not in clothing_from_person:
        corrections[label] = "tool"

for label in food_from_person:
    corrections[label] = "food"

for label in clothing_from_person:
    corrections[label] = "clothing"

for label in vehicles_from_tool:
    corrections[label] = "vehicle"

for label in furniture_from_tool:
    corrections[label] = "furniture"

for label in structures_from_tool:
    corrections[label] = "structure"

for label in food_from_structure:
    corrections[label] = "food"

for label in location_from_structure:
    corrections[label] = "location"

print(f"Total corrections to make: {len(corrections)}")
print(f"\nBreakdown:")
print(f"  → animal/living thing: {len(animals_from_person + animals_from_location)}")
print(f"  → tool: {len([l for l in corrections.values() if l == 'tool'])}")
print(f"  → vehicle: {len([l for l in corrections.values() if l == 'vehicle'])}")
print(f"  → furniture: {len([l for l in corrections.values() if l == 'furniture'])}")
print(f"  → structure: {len([l for l in corrections.values() if l == 'structure'])}")
print(f"  → food: {len([l for l in corrections.values() if l == 'food'])}")
print(f"  → location: {len([l for l in corrections.values() if l == 'location'])}")
print(f"  → clothing: {len([l for l in corrections.values() if l == 'clothing'])}")

Total corrections to make: 139

Breakdown:
  → animal/living thing: 19
  → tool: 12
  → vehicle: 71
  → furniture: 18
  → structure: 6
  → food: 9
  → location: 2
  → clothing: 2


In [66]:
# Apply corrections to semantic_data
corrected_data = {category: {"aliases": data["aliases"], "labels": []} for category, data in semantic_data.items()}

# First pass: move labels to correct categories
for category, data in semantic_data.items():
    for label in data["labels"]:
        if label in corrections:
            # Move to corrected category
            target_category = corrections[label]
            corrected_data[target_category]["labels"].append(label)
        else:
            # Keep in original category
            corrected_data[category]["labels"].append(label)

# Sort labels in each category
for category in corrected_data:
    corrected_data[category]["labels"] = sorted(set(corrected_data[category]["labels"]))

# Save corrected version
with open('SemanticLabels.json', 'w') as f:
    json.dump(corrected_data, f, indent=2)

print("✓ Applied corrections and saved to SemanticLabels.json")
print("\nUpdated category sizes:")
for category, data in corrected_data.items():
    print(f"  {category}: {len(data['labels'])} labels")

✓ Applied corrections and saved to SemanticLabels.json

Updated category sizes:
  person: 11 labels
  animal: 5 labels
  food: 43 labels
  vehicle: 71 labels
  location: 36 labels
  tool: 235 labels
  furniture: 18 labels
  color: 0 labels
  feeling: 2 labels
  establishment: 0 labels
  structure: 103 labels
  clothing: 46 labels
  living thing: 385 labels


In [67]:
# Verify: Extract labels after corrections and compare with before
labels_after = get_all_labels_from_semantic_json('SemanticLabels.json')

print(f"Labels before corrections: {len(labels_before)}")
print(f"Labels after corrections:  {len(labels_after)}")
print(f"\n✓ Same set of labels: {labels_before == labels_after}")

if labels_before != labels_after:
    missing = labels_before - labels_after
    added = labels_after - labels_before
    if missing:
        print(f"\n⚠ Labels missing after correction: {missing}")
    if added:
        print(f"\n⚠ Labels added after correction: {added}")
else:
    print("\n✓ Perfect! All labels preserved, just reorganized.")

Labels before corrections: 955
Labels after corrections:  955

✓ Same set of labels: True

✓ Perfect! All labels preserved, just reorganized.
